# Identificação de pivôs em Jussara-GO com Colab + Google Earth Engine

Notebook pronto para abrir no Google Colab.

Fluxo:
1. instalar/importar dependências;
2. autenticar no Google Earth Engine;
3. gerar o recorte municipal de Jussara-GO;
4. exportar um mosaico Sentinel-2 para o Google Drive;
5. treinar a U-Net usando `projects/sefazgogeoprocessamento/assets/pivo.zip` ou um ZIP no Drive.

In [ ]:
!pip install -q earthengine-api geemap tensorflow

In [ ]:
from pathlib import Path

import ee

PROJECT_ID = None  # opcional: informe o ID do projeto GEE/Google Cloud, se exigido pela conta.
DRIVE_FOLDER = "gee_jussara_pivos"
ZIP_PIVOS = "projects/sefazgogeoprocessamento/assets/pivo.zip"

## Autenticar no Earth Engine

Se o Colab pedir um projeto, preencha `PROJECT_ID` na célula anterior e rode novamente.

In [ ]:
ee.Authenticate()
if PROJECT_ID:
    ee.Initialize(project=PROJECT_ID)
else:
    ee.Initialize()

## Recorte de Jussara-GO e mosaico Sentinel-2

In [ ]:
def jussara_go_geometry() -> ee.Geometry:
    """Retorna o limite municipal de Jussara-GO usando a base GAUL nível 2."""
    municipios = ee.FeatureCollection("FAO/GAUL/2015/level2")
    jussara = (
        municipios.filter(ee.Filter.eq("ADM0_NAME", "Brazil"))
        .filter(ee.Filter.eq("ADM1_NAME", "Goias"))
        .filter(ee.Filter.eq("ADM2_NAME", "Jussara"))
    )
    return jussara.geometry()


def mask_s2_clouds(image: ee.Image) -> ee.Image:
    """Remove nuvens e cirrus usando a banda QA60 do Sentinel-2 SR Harmonized."""
    qa = image.select("QA60")
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000)


def sentinel2_jussara_composite(start_date: str, end_date: str) -> ee.Image:
    """Cria mosaico Sentinel-2 recortado para Jussara-GO com NDVI e NDWI."""
    aoi = jussara_go_geometry()
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 35))
        .map(mask_s2_clouds)
    )
    composite = collection.median().clip(aoi)
    ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = composite.normalizedDifference(["B3", "B8"]).rename("NDWI")
    return composite.select(["B2", "B3", "B4", "B8", "B11", "B12"]).addBands([ndvi, ndwi])

## Visualizar rapidamente o recorte no mapa

Essa célula ajuda a confirmar que Jussara-GO apareceu corretamente antes de exportar.

In [ ]:
import geemap

aoi = jussara_go_geometry()
image = sentinel2_jussara_composite("2024-05-01", "2024-10-31")

Map = geemap.Map()
Map.centerObject(aoi, 10)
Map.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0.02, "max": 0.3}, "Sentinel-2 RGB")
Map.addLayer(aoi, {}, "Jussara-GO")
Map

## Exportar o recorte para o Google Drive

In [ ]:
export_task = ee.batch.Export.image.toDrive(
    image=image,
    description="jussara_go_sentinel2_pivos_2024",
    folder=DRIVE_FOLDER,
    fileNamePrefix="jussara_go_sentinel2_pivos_2024",
    region=aoi,
    scale=10,
    maxPixels=1e13,
)
export_task.start()
print("Exportação iniciada. Acompanhe em: https://code.earthengine.google.com/tasks")
print("Task ID:", export_task.id)

## Treinar a U-Net com o ZIP de pivôs

Se o ZIP estiver no Google Drive, monte o Drive e ajuste `ZIP_PIVOS`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
# Exemplo se o ZIP estiver no Drive:
# ZIP_PIVOS = '/content/drive/MyDrive/caminho/para/pivo.zip'
print(f"ZIP configurado para treinamento: {Path(ZIP_PIVOS)}")

In [ ]:
if not Path('train_pivos_from_zip.py').exists():
    raise FileNotFoundError(
        'O arquivo train_pivos_from_zip.py não está no runtime do Colab. '
        'Abra o notebook a partir do repositório clonado ou faça upload do script antes de treinar.'
    )


In [ ]:
!python train_pivos_from_zip.py   --zip "$ZIP_PIVOS"   --epochs 40   --batch-size 8   --output Modelo_UNet_Pivos_Jussara.keras